In [ ]:
import torch
import torch.nn as nn
from stable_baselines3 import PPO
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.callbacks import BaseCallback
import numpy as np

from rl_setup import STEM_environment_history, simulation_parameters


# Auto focus: stable_baseline3 reinforcement learning

Author: Henry Bell, Updated 11/12/2025

# PPO Training Setup

In [ ]:

# Custom feature extractor 
class CustomCNN(BaseFeaturesExtractor):
    def __init__(self, observation_space, features_dim=256):
        super().__init__(observation_space, features_dim)
        
        n_frames = observation_space.spaces["images"].shape[0]
        h, w = observation_space.spaces["images"].shape[1:]
        n_actions = observation_space.spaces["actions"].shape[0]
        
        self.cnn = nn.Sequential(
            nn.Conv2d(n_frames, 16, kernel_size=4, stride=2),
            nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=3, stride=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=1),
            nn.ReLU(),
            nn.Flatten(),
        )
        
        with torch.no_grad():
            dummy = torch.zeros(1, n_frames, h, w)
            cnn_out_size = self.cnn(dummy).shape[1]
        
        self.action_mlp = nn.Sequential(
            nn.Linear(n_actions, 32),
            nn.ReLU(),
        )
        
        self.combined = nn.Sequential(
            nn.Linear(cnn_out_size + 32, features_dim),
            nn.ReLU(),
        )
    
    def forward(self, observations):
        images = observations["images"]
        actions = observations["actions"]
        
        cnn_features = self.cnn(images)
        action_features = self.action_mlp(actions)
        
        combined = torch.cat([cnn_features, action_features], dim=1)
        return self.combined(combined)

# Custom callback for logging
class DetailedLoggingCallback(BaseCallback):
    def __init__(self, print_freq=100, verbose=0):
        super().__init__(verbose)
        self.print_freq = print_freq
        self.episode_rewards = []
        self.episode_lengths = []
        
    def _on_step(self) -> bool:
        # Print every N steps
        if self.n_calls % self.print_freq == 0:
            # Get recent episode info if available
            if len(self.model.ep_info_buffer) > 0:
                recent_episodes = list(self.model.ep_info_buffer)[-10:]
                if recent_episodes:
                    mean_reward = np.mean([ep['r'] for ep in recent_episodes])
                    mean_length = np.mean([ep['l'] for ep in recent_episodes])
                    print(f"Step {self.n_calls:6d} | Episodes: {len(self.model.ep_info_buffer):4d} | "
                          f"Mean Reward (last 10): {mean_reward:7.2f} | "
                          f"Mean Length: {mean_length:5.1f}")
            else:
                print(f"Step {self.n_calls:6d} | Collecting experience...")
        
        return True

check_env(STEM_environment_history(config=simulation_parameters))


# Create model
policy_kwargs = dict(
    features_extractor_class=CustomCNN,
    features_extractor_kwargs=dict(features_dim=256),
)


model = PPO(
    "MultiInputPolicy",
    STEM_environment_history(config=simulation_parameters),
    policy_kwargs=policy_kwargs,
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.01,
    vf_coef=0.5,
    max_grad_norm=0.5,
    verbose=1,
    device="cuda",
)
print("Model created!\n")

# Create callback
logging_callback = DetailedLoggingCallback(print_freq=100)

# Train with callback
print("Starting training...")
print("-" * 80)
model.learn(
    total_timesteps=100000//5,
    callback=logging_callback,
    progress_bar=True  # This adds a progress bar!
)

print("\nTraining complete!")
# Save
model.save("ppo_stem_focusing")
print("Model saved to 'ppo_stem_focusing.zip'")

2025-11-12 21:44:38.448250: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-12 21:44:38.458210: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763012678.470844  581052 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763012678.475032  581052 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1763012678.485364  581052 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

Checking environment...


/home/hebell/miniconda3/envs/abtem_env/lib/python3.11/site-packages/stable_baselines3/common/env_checker.py:55: UserWarning: It seems that your observation images is an image but its `dtype` is (float32) whereas it has to be `np.uint8`. If your observation is not an image, we recommend you to flatten the observation to have only a 1D vector
  warnings.warn(
/home/hebell/miniconda3/envs/abtem_env/lib/python3.11/site-packages/stable_baselines3/common/env_checker.py:63: UserWarning: It seems that your observation space images is an image but the upper and lower bounds are not in [0, 255]. Because the CNN policy normalize automatically the observation you may encounter issue if the values are not in that range.
  warnings.warn(


Environment check passed!

Creating PPO model...
Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Model created!

Starting training...
--------------------------------------------------------------------------------


Output()

...

Step  11100 | Episodes:  100 | Mean Reward (last 10):    8.86 | Mean Length:  30.0

Step  11200 | Episodes:  100 | Mean Reward (last 10):    6.84 | Mean Length:  30.0

Step  11300 | Episodes:  100 | Mean Reward (last 10):    5.90 | Mean Length:  30.0

Step  11400 | Episodes:  100 | Mean Reward (last 10):    6.36 | Mean Length:  30.0

KeyboardInterrupt: 

# SAC Training Setup

In [1]:
from stable_baselines3 import SAC
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv

# Create env (single env for SAC) and wrap for logging
def make_env():
    return Monitor(STEM_environment_history(config=simulation_parameters))

env = make_env()

# Policy kwargs reuse your CustomCNN feature extractor
policy_kwargs = dict(
    features_extractor_class=CustomCNN,
    features_extractor_kwargs=dict(features_dim=256),
)

print("Creating SAC model...")
model = SAC(
    "MultiInputPolicy",
    env,
    policy_kwargs=policy_kwargs,
    buffer_size=200_000,       # large replay buffer
    learning_starts=2000,      # collect this many transitions with random actions first
    batch_size=256,
    tau=0.005,
    gamma=0.99,
    train_freq=1,              # train every step
    gradient_steps=1,
    ent_coef='auto',           # automatic entropy tuning
    learning_rate=3e-4,
    verbose=1,
    device="cuda",
)
print("SAC model created!\n")

#prefill replay buffer with random transitions 
PREFILL_STEPS = 2000
if PREFILL_STEPS > 0:
    print(f"Prefilling replay buffer with {PREFILL_STEPS} random steps...")
    try:
        obs = env.reset()
        for _ in range(PREFILL_STEPS):
            action = env.action_space.sample()
            next_obs, reward, done, info = env.step(action)
            # try to add single transition to the (possibly Dict) replay buffer
            try:
                model.replay_buffer.add(obs, next_obs, action, reward, done, info)
            except Exception:
                # fallback: if add fails (shapes / vec env mismatch), just step env to let SAC collect
                pass
            obs = next_obs if not done else env.reset()
        print("Prefill done (best-effort).")
    except Exception as e:
        print("Prefill failed or not supported in this env; continuing to training. Error:", e)

# Train
model.learn(
    total_timesteps=100000,
    log_interval=10,
    progress_bar=True,
)

# Save
model.save("sac_stem_focusing")


2025-11-12 23:53:50.044673: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-12 23:53:50.054797: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763020430.067572  625497 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763020430.071760  625497 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1763020430.082324  625497 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

KeyboardInterrupt: 